# 12 — IS-Only Re-Selection

**Author:** Sacha Huberty · IE MBDS, Algorithmic Trading
**Purpose:** Thin driver reproducing the re-selection task's results from
`src/atlas` (`selection.py`, `reselection.py`) and the committed artifacts
in `reports/results/`. Does **not** re-run the multi-hour IS walk-forward
or the Phase 4 OOS backtests — those were run once, checkpointed, and
their outputs committed; this notebook loads and displays them so the
whole task is reproducible in seconds from a clean checkout, not hours.
Full narrative, every table's interpretation, and the final
recommendation live in `docs/RESELECTION.md`; this notebook is evidence
that the numbers quoted there are real, not a report in its own right.
**Last updated:** 2026-11-01.


## Setup

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.options.display.float_format = '{:.4f}'.format

from atlas import data, selection

PROJECT_ROOT = data.PROJECT_ROOT
RESULTS_DIR = PROJECT_ROOT / "reports" / "results"


## Data

Load the committed result artifacts from Phases 2-4.

In [2]:
phase2_grid = pd.read_csv(RESULTS_DIR / "reselection_phase2_grid.csv")
with open(RESULTS_DIR / "reselection_phase2_decisions.json") as f:
    phase2_decisions = json.load(f)
phase3_sensitivity = pd.read_csv(RESULTS_DIR / "reselection_phase3_sensitivity.csv")
with open(RESULTS_DIR / "reselection_phase4_statistics.json") as f:
    phase4_stats = json.load(f)

print("Phase 2 grid rows:", len(phase2_grid), "(expect 144)")
print("Phase 3 sensitivity rows:", len(phase3_sensitivity), "(expect 12)")
print("Phase 4 windows:", list(phase4_stats["windows"].keys()))


Phase 2 grid rows: 144 (expect 144)
Phase 3 sensitivity rows: 12 (expect 12)
Phase 4 windows: ['OOS_2022_2026', 'full_2014_2026']


## Analysis

### Pre-registered fold scheme and stability score (src/atlas/selection.py)

The same `annual_fold_periods` + `stability_score` primitives notebook 09's
original meanreversion/momentum grid searches used, extracted and tested
(`tests/test_selection.py`) so this task reuses them exactly rather than
inventing a new fold scheme -- see `docs/RESELECTION_PREREG.md` Sec 1.

In [3]:
is_folds_example = selection.annual_fold_periods(
    pd.bdate_range("2010-01-01", "2021-12-31")
)
print("IS annual folds:", list(is_folds_example))
print("n folds:", len(is_folds_example))


IS annual folds: [Period('2010', 'Y-DEC'), Period('2011', 'Y-DEC'), Period('2012', 'Y-DEC'), Period('2013', 'Y-DEC'), Period('2014', 'Y-DEC'), Period('2015', 'Y-DEC'), Period('2016', 'Y-DEC'), Period('2017', 'Y-DEC'), Period('2018', 'Y-DEC'), Period('2019', 'Y-DEC'), Period('2020', 'Y-DEC'), Period('2021', 'Y-DEC')]
n folds: 12


### Phase 2: the 144-cell IS grid and the one-setting-at-a-time decisions

In [4]:
current_row = phase2_grid[
    (phase2_grid.v3 == False) & (phase2_grid.v5 == False)
    & (phase2_grid.risk_off_target == "legacy") & (phase2_grid.gate_a == 2)
    & (phase2_grid.turnover == "2pct_0.5pct")
]
current_row[["stability_score", "excess_sharpe_2014_2021", "ann_vol_2014_2021",
             "max_drawdown_2014_2021", "avg_weekly_turnover"]]


,stability_score,excess_sharpe_2014_2021,ann_vol_2014_2021,max_drawdown_2014_2021,avg_weekly_turnover
4,-0.3327,0.4931,0.1002,-0.2370,0.0111


In [5]:
decisions_summary = pd.DataFrame(
    {k: {
        "adopt_candidate": v["adopt_candidate"],
        "current_stability": round(v["current_stability"], 4),
        "candidate_stability": round(v["candidate_stability"], 4),
        "fold_win_count": v["fold_win_count"],
        "n_folds": v["n_folds"],
    } for k, v in phase2_decisions.items()}
).T
decisions_summary


,adopt_candidate,current_stability,candidate_stability,fold_win_count,n_folds
V3,False,-0.3327,-0.1669,6,12
V5,True,-0.3327,-0.1835,7,12
risk_off_bonds_cash,False,-0.3327,-0.3334,3,12
risk_off_bonds_gold,False,-0.3327,-0.3332,4,12
gate_A_1,False,-0.3327,-0.3156,3,12
gate_A_3,False,-0.3327,-0.2635,6,12
gate_A_5,False,-0.3327,-0.4009,6,12
turnover_1pct_0.5pct,True,-0.3327,-0.3230,8,12
turnover_4pct_0.5pct,False,-0.3327,-0.3427,4,12


In [6]:
joint_best = phase2_grid.sort_values("stability_score", ascending=False).iloc[0]
print("Full-grid joint best cell (NOT adopted -- see PREREG Sec 3 caveat):")
print(joint_best[["v3", "v5", "risk_off_target", "gate_a", "turnover", "stability_score"]])


Full-grid joint best cell (NOT adopted -- see PREREG Sec 3 caveat):
v3                        True
v5                        True
risk_off_target         legacy
gate_a                       5
turnover           4pct_0.5pct
stability_score        -0.0134
Name: 119, dtype: object


### Phase 3: sensitivity surface (diagnostic only, nothing adopted)

In [7]:
phase3_sensitivity.pivot(
    index="value", columns="parameter",
    values="excess_sharpe_2014_2021",
)


parameter,delta,per_asset_cap,regime_view_magnitude,tau
value,,,,
0.0150,NaN,NaN,0.4815,NaN
0.0250,NaN,NaN,NaN,0.4886
0.0300,NaN,NaN,0.4886,NaN
0.0500,NaN,NaN,0.4975,0.4886
0.1000,NaN,NaN,NaN,0.4886
0.1500,NaN,0.5227,NaN,NaN
0.2000,NaN,0.4886,NaN,NaN
0.2500,NaN,0.4914,NaN,NaN
1.5000,0.5144,NaN,NaN,NaN


## Results

### Phase 4: the single OOS look

In [8]:
rows = []
for window, metrics_by_cfg in phase4_stats["windows"].items():
    for cfg_key in ("a_current", "b_phase2", "c_permanent"):
        m = metrics_by_cfg[cfg_key]
        rows.append({
            "window": window, "config": cfg_key,
            "ann_return": m["ann_return"], "ann_vol": m["ann_vol"],
            "excess_sharpe": m["excess_sharpe"],
            "max_drawdown": m["max_drawdown"], "sharpe_se": m["sharpe_se"],
        })
pd.DataFrame(rows).set_index(["window", "config"])


ann_return  ann_vol  excess_sharpe  max_drawdown  \
window         config                                                          
OOS_2022_2026  a_current        0.0998   0.1052         0.5791       -0.1589   
               b_phase2         0.0813   0.0934         0.4579       -0.1271   
               c_permanent      0.0942   0.0872         0.6199       -0.1339   
full_2014_2026 a_current        0.0702   0.1012         0.5292       -0.2315   
               b_phase2         0.0594   0.0844         0.4965       -0.2048   
               c_permanent      0.0594   0.0823         0.5072       -0.1966   

                            sharpe_se  
window         config                  
OOS_2022_2026  a_current       0.4923  
               b_phase2        0.4788  
               c_permanent     0.4974  
full_2014_2026 a_current       0.2982  
               b_phase2        0.2960  
               c_permanent     0.2967

In [9]:
paired_rows = []
for window, metrics_by_cfg in phase4_stats["windows"].items():
    for pair_key in ("paired_b_vs_c", "paired_a_vs_b"):
        t = metrics_by_cfg[pair_key]
        paired_rows.append({
            "window": window, "comparison": pair_key,
            "mean_diff": t["mean"], "tstat": t["tstat"],
            "pvalue": t["pvalue"], "n": t["n"], "maxlags": t["maxlags"],
        })
pd.DataFrame(paired_rows).set_index(["window", "comparison"])


mean_diff   tstat  pvalue    n  maxlags
window         comparison                                            
OOS_2022_2026  paired_b_vs_c    -0.0002 -0.6164  0.5376  252        4
               paired_a_vs_b     0.0003  0.8424  0.3996  252        4
full_2014_2026 paired_b_vs_c     0.0000  0.0243  0.9806  670        6
               paired_a_vs_b     0.0002  0.8288  0.4072  670        6

In [10]:
print("N_trials (Phase 2 pre-registered grid):", phase4_stats["n_trials_prereg"])
print("N prior OOS looks + this task's own post-hoc decision:", phase4_stats["n_trials_total_171"] - phase4_stats["n_trials_prereg"])
print("N_trials total (as originally reported):", phase4_stats["n_trials_total"])
print("N_trials total (headline, corrected -- see erratum below):", phase4_stats["n_trials_headline"])
print("Empirical sharpe_std_across_trials:", round(phase4_stats["sharpe_std_across_trials"], 4))
print()
for window, metrics_by_cfg in phase4_stats["windows"].items():
    m = metrics_by_cfg["b_phase2"]
    print(
        window,
        "DSR weekly N=170:", round(m["deflated_sharpe_ratio_weekly_n170"], 4),
        "| N=171 (headline):", round(m["deflated_sharpe_ratio_weekly_n171"], 4),
        "| PSR (no trial correction):",
        round(m["probabilistic_sharpe_ratio_weekly_no_trial_correction"], 4),
    )
    print(
        "   superseded (units-bug) values -- kept for the record, not deleted:",
        "daily:", round(m["superseded_dsr_units_bug"]["deflated_sharpe_ratio_daily_n_obs"], 4),
        "| weekly:", round(m["superseded_dsr_units_bug"]["deflated_sharpe_ratio_weekly_n_obs"], 4),
    )


N_trials (Phase 2 pre-registered grid): 144
N prior OOS looks + this task's own post-hoc decision: 27
N_trials total (as originally reported): 170
N_trials total (headline, corrected -- see erratum below): 171
Empirical sharpe_std_across_trials: 0.116

OOS_2022_2026 DSR weekly N=170: 0.6235 | N=171 (headline): 0.6233 | PSR (no trial correction): 0.8425
   superseded (units-bug) values -- kept for the record, not deleted: daily: 1.0 | weekly: 0.9846
full_2014_2026 DSR weekly N=170: 0.7428 | N=171 (headline): 0.7425 | PSR (no trial correction): 0.9624
   superseded (units-bug) values -- kept for the record, not deleted: daily: 1.0 | weekly: 1.0


## Notes / next steps

- **Recommendation: keep the current config.** This is a deviation
  from the pre-registered rule above (which would adopt V5 and the 1%
  turnover cap) -- decided on **IS-only grounds**, not because of the
  OOS numbers in this notebook: V5's apparent IS fold-win edge comes
  entirely from the 2010-2013 cold-start years and loses, on average,
  over 2014-2021; the 1% cap's IS edge is noise-sized against the
  fold-to-fold variance. The OOS table above is NOT the stated reason
  -- a decision reasoned from the window being treated as
  contaminated would itself be contaminated. Full reasoning, the
  post-hoc IS-only diagnostic table, and the cold-start mechanism are
  in `docs/RESELECTION.md` Sections 4-6.
- **Erratum (external review):** the DSR values originally shown
  above mixed an annualized Sharpe ratio with a per-observation
  `n_obs` -- a units bug, not a formula bug. The corrected,
  periods-per-year-consistent values are now printed in the cell
  above (weekly DSR ~0.62 OOS / ~0.74 full, N=171 headline); the old
  (superseded) ~0.98 / ~1.0 values are printed alongside them for the
  record, not deleted. See `src/atlas/metrics.py` and
  `docs/RESELECTION.md` Section 7.
- `config/config.yaml`, `config/regime_posture.yaml`, and `live/` were
  never modified by this task, including this correction pass.
- The expensive IS walk-forward (shared cache: 626 decisions,
  2010-2021; 12 view-variant caches; the 144-cell grid; two full
  2010-2026 OOS backtests with the real anomaly autoencoder) was run
  once via scratch driver scripts (not committed -- the reusable logic
  lives in `src/atlas/selection.py`/`reselection.py`, tested) and its
  results committed to `reports/results/`, which is what this notebook
  reproduces from.
